# EDA on Automobiles Dataset

## Project Overview
Exploratory Data Analysis on the UCI Automobiles dataset. This notebook covers data loading, cleaning, preprocessing, and initial insights.

## 1. Import Libraries

In [ ]:
import requests
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 2. Download Data

In [ ]:
# Download the dataset from UCI repository
url = "http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data"
print("Downloading data...")
res = requests.get(url)
print(f"Status code: {res.status_code}")
print(f"Data size: {len(res.content)} bytes")

In [ ]:
# Load into pandas DataFrame
auto = pd.read_csv(io.StringIO(res.decode("utf-8")), header=None)

# Assign column names
auto.columns = ['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration', 
                'num-of-doors', 'body-style', 'drive-wheels', 'engine-location', 
                'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-type', 
                'num-of-cylinders', 'engine-size', 'fuel-system', 'bore', 'stroke', 
                'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 
                'price']

print(f"Dataset shape: {auto.shape}")
auto.head()

In [ ]:
# Quick info about the dataset
auto.info()

## 3. Data Cleaning

### 3.1 Handle Missing Values (represented as '?')

In [ ]:
# Replace '?' with NaN for proper handling
auto = auto.replace('?', np.nan)

# Check initial missing values
print("Missing values before cleaning:")
print(auto.isnull().sum()[auto.isnull().sum() > 0].sort_values(ascending=False))

In [ ]:
# Drop rows where price is missing (target variable)
initial_shape = auto.shape
auto.dropna(subset=['price'], inplace=True)
print(f"Dropped {initial_shape[0] - auto.shape[0]} rows with missing price")
print(f"New shape: {auto.shape}")

In [ ]:
# Check missing values after dropping price
print(auto.isnull().sum().sort_values(ascending=False))

### 3.2 Drop 'normalized-losses' column (too many missing values)

In [ ]:
# Drop the column with most missing values
auto.drop(columns=['normalized-losses'], inplace=True)
print(f"Remaining columns: {auto.columns.tolist()}")

### 3.3 Convert numeric columns and fill NaNs with median

In [ ]:
# Columns that should be numeric
numeric_columns = ['wheel-base', 'length', 'width', 'height', 'curb-weight', 
                   'engine-size', 'bore', 'stroke', 'compression-ratio', 
                   'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'price']

# Convert to numeric (coerce errors to NaN)
for col in numeric_columns:
    auto[col] = pd.to_numeric(auto[col], errors='coerce')

# Fill numeric NaNs with median (robust to outliers)
numeric_cols = auto.select_dtypes(include='number').columns
auto[numeric_cols] = auto[numeric_cols].fillna(auto[numeric_cols].median())

print("Numeric columns filled with median values")

In [ ]:
# Verify no missing values remain in numeric columns
print(auto[numeric_cols].isnull().sum())

### 3.4 Fill categorical NaNs with mode

In [ ]:
# Fill categorical NaNs with mode (most frequent value)
categorical_cols = auto.select_dtypes(include='object').columns
for col in categorical_cols:
    mode_value = auto[col].mode()
    if len(mode_value) > 0:
        auto[col] = auto[col].fillna(mode_value.iloc[0])
    else:
        auto[col] = auto[col].fillna('unknown')

print("Categorical columns filled with mode values")

In [ ]:
# Final check for missing values
print("\nRemaining missing values after cleaning:")
print(auto.isnull().sum().sum())
print("All missing values handled!")

### 3.5 Remove duplicates

In [ ]:
# Check for and remove duplicate rows
duplicates_before = auto.duplicated().sum()
auto.drop_duplicates(inplace=True)
print(f"Removed {duplicates_before} duplicate rows")
print(f"Final dataset shape: {auto.shape}")

## 4. Data Type Verification

In [ ]:
# Verify data types after cleaning
auto.dtypes

In [ ]:
# Summary statistics for numeric columns
auto[numeric_cols].describe()

## 5. Categorical Variables Encoding (One-Hot Encoding)

In [ ]:
# Define categorical columns for one-hot encoding
categorical_cols = ['make', 'fuel-type', 'aspiration', 'num-of-doors',
                    'body-style', 'drive-wheels', 'engine-location',
                    'engine-type', 'num-of-cylinders', 'fuel-system']

# Verify these columns exist
existing_cat_cols = [col for col in categorical_cols if col in auto.columns]
print(f"Encoding {len(existing_cat_cols)} categorical columns: {existing_cat_cols}")

In [ ]:
# Apply one-hot encoding with drop_first to avoid multicollinearity
auto_encoded = pd.get_dummies(auto, columns=existing_cat_cols, drop_first=True)

print(f"Shape before encoding: {auto.shape}")
print(f"Shape after encoding: {auto_encoded.shape}")
print(f"Added {auto_encoded.shape[1] - auto.shape[1]} dummy columns")

In [ ]:
# Display first few rows of encoded dataset
auto_encoded.head()

In [ ]:
# Final dataset info
print("=" * 50)
print("FINAL CLEANED AND ENCODED DATASET")
print("=" * 50)
print(f"Total samples: {len(auto_encoded)}")
print(f"Total features: {len(auto_encoded.columns)}")
print(f"\nFeature list:\n{auto_encoded.columns.tolist()}")

## 6. Save Cleaned Dataset (Optional)

In [ ]:
# Save to CSV for future use
auto_encoded.to_csv('automobiles_cleaned.csv', index=False)
print("Dataset saved as 'automobiles_cleaned.csv'")

## 7. Initial Data Exploration

In [ ]:
# Check target variable distribution (price)
print("Price Statistics:")
print(auto_encoded['price'].describe())

# Plot price distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(auto_encoded['price'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Car Prices')

axes[1].boxplot(auto_encoded['price'])
axes[1].set_ylabel('Price')
axes[1].set_title('Boxplot of Car Prices')

plt.tight_layout()
plt.show()

In [ ]:
# Check correlations with price (top 10)
price_corr = auto_encoded.corr()['price'].sort_values(ascending=False)
print("Top 10 features correlated with price:")
print(price_corr[1:11])  # Skip price itself

In [ ]:
# Print summary of cleaning steps
print("=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)
print(f"✓ Downloaded from: {url}")
print(f"✓ Removed rows with missing price values")
print(f"✓ Dropped 'normalized-losses' column (high missing rate)")
print(f"✓ Filled numeric NaNs with median")
print(f"✓ Filled categorical NaNs with mode")
print(f"✓ Removed {duplicates_before} duplicate rows")
print(f"✓ One-hot encoded {len(existing_cat_cols)} categorical variables")
print(f"✓ Final dataset: {len(auto_encoded)} rows × {len(auto_encoded.columns)} columns")
print("=" * 60)